# Train the casting-photo classifier (Colab)
Set **Runtime → Change runtime type → GPU** first. See `docs/TRAINING_GUIDE.md` for background and how to read the report.
Run the cells top to bottom.

In [ ]:
!nvidia-smi | head -15   # confirms a GPU is attached (if this fails, training will be very slow)

## 1. Upload the project
Zip the project folder (the one containing `backend/`, `ml/`, `tools/`) and upload it.

In [ ]:
from google.colab import files
up = files.upload()          # choose defect-detection-app.zip
!unzip -q -o defect-detection-app.zip
%cd defect-detection-app
!pip -q install numpy opencv-python-headless pillow

## 2. Get the dataset
Either upload your Kaggle API token (`kaggle.json`: Kaggle → Settings → Create New Token) or upload the dataset zip yourself.

In [ ]:
from google.colab import files
print('Upload kaggle.json'); files.upload()
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip -q install kaggle
!kaggle datasets download -d ravirajsinh45/real-life-industrial-dataset-of-casting-product -p casting_zip --unzip
!find casting_zip -maxdepth 3 -type d | head

## 3. Smoke test (a couple of minutes; the model it produces is NOT usable)

In [ ]:
!python -m ml.train_model --data-dir casting_zip --pass-classes ok_front --input-mode image --smoke-test --weights none --out-dir /tmp/smoke

## 4. Real training
Check the printed layout is `official`, classes are `['def_front', 'ok_front']`, and there is no `LEAKAGE` error.

In [ ]:
!python -m ml.train_model --data-dir casting_zip --pass-classes ok_front --input-mode image --model-name casting-photo-v1 --out-dir models

## 5. Read the report, then download the model
Look at **recall of `def_front`** and the **threshold table** before using the model.

In [ ]:
print(open('models/evaluation_report.md').read())

In [ ]:
!zip -qr trained_model.zip models
from google.colab import files
files.download('trained_model.zip')   # copy defect_model.keras + labels.json into the app's models/ folder